# Predict Customer Churn — CatBoost Tuned Multi-Seed + Feature Engineering

CatBoost's key advantage over XGBoost/LightGBM on this dataset:
- **Native categorical handling** — raw string columns passed directly, no label encoding
- **Ordered boosting** — reduces prediction shift and overfitting on small-medium datasets
- **Built-in NaN handling** — no imputation needed for numeric columns

Pipeline:
1. **Feature Engineering** — same interaction features as XGBoost notebook
2. **Optuna Tuning** — `catboost.cv()` inside each trial (robust, no single-split bias)
3. **Multi-Seed Ensemble** — 5 seeds × 10 folds = 50 models
4. **GPU Support** — `task_type='GPU'` auto-injected when GPU detected

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from catboost import cv as catboost_cv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import optuna
from tqdm.auto import tqdm
import warnings
import subprocess
import os
import gc

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Environment Detection ────────────────────────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle/input')

def has_local_gpu():
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, timeout=5)
        return result.returncode == 0
    except Exception:
        return False

USE_GPU  = IS_KAGGLE or has_local_gpu()
DATA_DIR = '/kaggle/input/playground-series-s6e3/' if IS_KAGGLE else '../data/'

print(f"Environment  : {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"GPU          : {'Enabled ✓' if USE_GPU else 'Disabled — using CPU'}")
print(f"Data dir     : {DATA_DIR}")

import catboost
print(f"CatBoost ver : {catboost.__version__}")

# ── Experiment Settings ───────────────────────────────────────────────────────
RUN_TUNING   = True
N_TRIALS     = 50
N_CV_FOLDS   = 5
SEEDS        = [42, 2024, 777, 1337, 999]
N_SPLITS     = 10
TOTAL_MODELS = len(SEEDS) * N_SPLITS

# Pre-computed fallback (used when RUN_TUNING = False)
PRECOMPUTED_PARAMS = {
    'learning_rate':   0.05,
    'depth':           6,
    'l2_leaf_reg':     3.0,
    'subsample':       0.8,
    'colsample_bylevel': 0.8,
    'min_data_in_leaf': 5,
}

## 1. Feature Engineering

In [ ]:
def engineer_features(df):
    """Row-level statistics + domain interaction features on numeric columns."""
    df = df.copy()
    num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

    df['num_sum']  = df[num_cols].sum(axis=1)
    df['num_mean'] = df[num_cols].mean(axis=1)
    df['num_std']  = df[num_cols].std(axis=1)
    df['num_max']  = df[num_cols].max(axis=1)
    df['num_min']  = df[num_cols].min(axis=1)

    df['Average_Monthly_Actual'] = df['TotalCharges'] / (df['tenure'] + 1e-5)
    df['Monthly_diff']           = df['MonthlyCharges'] - df['Average_Monthly_Actual']
    df['Monthly_ratio']          = df['MonthlyCharges'] / (df['Average_Monthly_Actual'] + 1e-5)

    return df

print('Feature engineering function defined.')

## 2. Load & Preprocess Data

CatBoost handles raw string categoricals and NaN values natively — **no label encoding or numeric imputation required**.
We only fill categorical NaNs with `'Missing'` so CatBoost treats them as a valid category.

In [ ]:
print(f'Loading data from {DATA_DIR} ...')
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sub      = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

TARGET = 'Churn'
if train_df[TARGET].dtype == 'object':
    train_df[TARGET] = train_df[TARGET].map({'Yes': 1, 'No': 0, '1': 1, '0': 0})

features = [c for c in train_df.columns if c not in ['id', TARGET]]

# Identify categorical columns (keep as strings — CatBoost handles them natively)
cat_features = [c for c in features if train_df[c].dtype == 'object']
num_features = [c for c in features if c not in cat_features]

print(f'Categorical features ({len(cat_features)}): {cat_features}')
print(f'Numeric features    ({len(num_features)}): {num_features}')

# Fill categorical NaNs with 'Missing' so CatBoost sees them as a valid category
for col in tqdm(cat_features, desc='Filling cat NaNs'):
    train_df[col] = train_df[col].fillna('Missing').astype(str)
    test_df[col]  = test_df[col].fillna('Missing').astype(str)

# Feature engineering
print('\nEngineering features...')
X      = engineer_features(train_df[features])
X_test = engineer_features(test_df[features])
y      = train_df[TARGET]

# Update cat_features list to include any new engineered columns that are strings
# (engineered features here are all numeric, so cat_features list stays the same)
print(f'\nTrain shape  : {X.shape}')
print(f'Test shape   : {X_test.shape}')
print(f'Target dist  : {y.value_counts().to_dict()}')
print(f'Cat features : {cat_features}')
print(f'New features : {[c for c in X.columns if c not in features]}')

## 3. Optuna Hyperparameter Tuning

`catboost.cv()` runs a full stratified CV **per trial** — same robust approach as the XGBoost notebook.

In [ ]:
# ── Speed optimisations ───────────────────────────────────────────────────────
# grow_policy='Lossguide'  → leaf-wise growth (like LightGBM), much faster than
#                            CatBoost's default symmetric tree on GPU
# bootstrap_type='Bernoulli' + subsample → faster than Bayesian bootstrap
# border_count=64          → fewer numeric bins; 64 is enough for most tabular data
# od_type='Iter' + od_wait → simpler early-stopping check, lower overhead
# colsample_bylevel (rsm)  → NOT supported on GPU — omitted when USE_GPU=True

TUNE_ITERATIONS     = 1000
ENSEMBLE_ITERATIONS = 3000

base_params = {
    'loss_function':  'Logloss',
    'eval_metric':    'AUC',
    'task_type':      'GPU' if USE_GPU else 'CPU',
    'verbose':        False,
    'random_seed':    42,
    'grow_policy':    'Lossguide',
    'max_leaves':     64,
    'bootstrap_type': 'Bernoulli',
    'border_count':   64,
    'od_type':        'Iter',
    'od_wait':        50,
}

print(f"task_type    : {base_params['task_type']}")
print(f"grow_policy  : {base_params['grow_policy']}  (leaf-wise = faster)")
print(f"colsample_bylevel (rsm) : {'DISABLED — not supported on GPU' if USE_GPU else 'ENABLED'}")

# Full train Pool — built once, reused across all Optuna trials
train_pool_full = Pool(X, label=y, cat_features=cat_features)

if RUN_TUNING:
    print(f'\nStarting Optuna tuning: {N_TRIALS} trials × {N_CV_FOLDS}-fold CV each')

    pbar = tqdm(total=N_TRIALS, desc='Optuna Trials', unit='trial')
    best_value_so_far = [0.0]

    def objective(trial):
        params = {
            **base_params,
            'iterations':       TUNE_ITERATIONS,
            'learning_rate':    trial.suggest_float('learning_rate',    0.03,  0.3,  log=True),
            'depth':            trial.suggest_int(  'depth',            4,     8),
            'l2_leaf_reg':      trial.suggest_float('l2_leaf_reg',      1e-3,  10.0, log=True),
            'subsample':        trial.suggest_float('subsample',        0.6,   1.0),
            'min_data_in_leaf': trial.suggest_int(  'min_data_in_leaf', 1,     50),
            'random_strength':  trial.suggest_float('random_strength',  1e-3,  5.0,  log=True),
        }
        # colsample_bylevel / rsm is CPU-only in CatBoost (non-pairwise modes)
        if not USE_GPU:
            params['colsample_bylevel'] = trial.suggest_float('colsample_bylevel', 0.6, 1.0)

        cv_result = catboost_cv(
            pool       = train_pool_full,
            params     = params,
            fold_count = N_CV_FOLDS,
            stratified = True,
            seed       = 42,
            verbose    = False,
        )

        score     = cv_result['test-AUC-mean'].max()
        best_iter = int(cv_result['test-AUC-mean'].idxmax())
        trial.set_user_attr('best_iteration', best_iter)

        if score > best_value_so_far[0]:
            best_value_so_far[0] = score
        pbar.set_postfix({'AUC': f'{score:.5f}', 'Best': f'{best_value_so_far[0]:.5f}'})
        pbar.update(1)
        return score

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=N_TRIALS)
    pbar.close()

    best_trial = study.best_trial
    best_iter  = best_trial.user_attrs.get('best_iteration', 500)

    print(f"\n{'='*55}")
    print(f"Best CV AUC    : {best_trial.value:.5f}")
    print(f"Best iteration : {best_iter}")
    print(f"Best params    :")
    for k, v in best_trial.params.items():
        print(f"  {k:25s}: {v}")
    print(f"{'='*55}")

    best_params = {**base_params, **best_trial.params}

    study.trials_dataframe().to_csv('catboost_tuning_results.csv', index=False)
    print('\nAll trial results saved → catboost_tuning_results.csv')

else:
    print('Skipping tuning — using PRECOMPUTED_PARAMS.')
    best_params = {**base_params, **PRECOMPUTED_PARAMS}
    best_iter   = 500
    for k, v in PRECOMPUTED_PARAMS.items():
        print(f'  {k:25s}: {v}')

## 4. Multi-Seed Ensemble

In [ ]:
print(f'Multi-Seed Ensemble: {len(SEEDS)} seeds × {N_SPLITS} folds = {TOTAL_MODELS} models\n')

# Ensemble uses full iteration budget with best tuned params
ensemble_params = {
    **best_params,
    'iterations':  ENSEMBLE_ITERATIONS,
    'od_wait':     150,
}

test_preds_sum = np.zeros(len(X_test))
global_oof_sum = np.zeros(len(X))
fold_auc_log   = []

outer_bar = tqdm(SEEDS, desc='Seeds', position=0)

for seed in outer_bar:
    outer_bar.set_description(f'Seed {seed}')
    skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    seed_oof = np.zeros(len(X))

    inner_bar = tqdm(
        enumerate(skf.split(X, y)),
        total    = N_SPLITS,
        desc     = '  Folds',
        position = 1,
        leave    = False,
    )

    for fold, (train_idx, val_idx) in inner_bar:
        X_tr,  X_val  = X.iloc[train_idx],  X.iloc[val_idx]
        y_tr,  y_val  = y.iloc[train_idx],  y.iloc[val_idx]

        train_pool = Pool(X_tr,  label=y_tr,  cat_features=cat_features)
        val_pool   = Pool(X_val, label=y_val, cat_features=cat_features)
        test_pool  = Pool(X_test,             cat_features=cat_features)

        model = CatBoostClassifier(**{**ensemble_params, 'random_seed': seed})
        model.fit(train_pool, eval_set=val_pool)

        val_preds  = model.predict_proba(val_pool)[:, 1]
        test_preds = model.predict_proba(test_pool)[:, 1]

        fold_auc = roc_auc_score(y_val, val_preds)
        fold_auc_log.append(fold_auc)

        seed_oof[val_idx]        = val_preds
        global_oof_sum[val_idx] += val_preds
        test_preds_sum          += test_preds

        inner_bar.set_postfix({'fold_auc': f'{fold_auc:.5f}', 'best_iter': model.best_iteration_})

        del model, train_pool, val_pool, test_pool
        gc.collect()

    seed_auc = roc_auc_score(y, seed_oof)
    outer_bar.set_postfix({'seed_oof_auc': f'{seed_auc:.5f}'})

# ── Final metrics ─────────────────────────────────────────────────────────────
global_oof = global_oof_sum / len(SEEDS)
final_auc  = roc_auc_score(y, global_oof)

print(f"\n{'='*55}")
print(f"Fold AUC — mean : {np.mean(fold_auc_log):.5f}  std: {np.std(fold_auc_log):.5f}")
print(f"Global Ensemble OOF AUC : {final_auc:.5f}")
print(f"{'='*55}")

## 5. Generate Submission

In [ ]:
final_test_preds = test_preds_sum / TOTAL_MODELS
sub[TARGET] = final_test_preds

out_file = 'submission_catboost_tuned_multiseed_fe.csv'
sub.to_csv(out_file, index=False)

print(f'Submission saved → {out_file}')
print(f'Prediction range : [{final_test_preds.min():.4f}, {final_test_preds.max():.4f}]')
display(sub.head())